# Create Dashboard using Plotly and Dash – Final Assignment Part 2

**CISC7204 Assignment 02 – Module 05 Final Project Lab 02**

## Scenario
The objective of this part of the Final Assignment is to analyze the historical trends in automobile sales during recession periods. The goal is to provide insights into how the sales of XYZAutomotives were affected during times of recession, and to demonstrate **Dashboarding** skills using *Plotly and Dash*.

The dashboard represents two main reports:
- **Yearly Automobile Sales Statistics**
- **Recession Period Statistics**

> Note: Year range is between 1980 and 2013.

## Dataset
[automobile-sales.csv](automobile-sales.csv) – local file in the same directory as this notebook.

## Components of the Report Items
**1. Yearly Automobile Sales Statistics**
- Yearly Automobile Sales Using Line Chart for the Whole Period
- Total Monthly Automobile Sales Using Line Chart
- Average Vehicles Sold by Vehicle Type in the Selected Year (bar chart)
- Total Advertisement Expenditure for Each Vehicle Using Pie Chart

**2. Recession Period Statistics**
- Average Automobile Sales Fluctuation Over Recession Period (Year-wise)
- Average Number of Vehicles Sold by Vehicle Type
- Total Expenditure Share by Vehicle Type During Recessions
- Effect of Unemployment Rate on Vehicle Type and Sales

## REVIEW – Import the required libraries and load the data

In [1]:
import dash
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
from dash import dcc
from dash import html
from dash.dependencies import Input, Output

# Load the data using pandas (local file in the same directory as this notebook)
data = pd.read_csv('automobile-sales.csv')
print('Data loaded. Shape:', data.shape)

# Initialize the Dash app
app = dash.Dash(__name__)

Data loaded. Shape: (2112, 15)


## TASK 2.1 – Create a Dash application and give it a meaningful title

Provide the title of the dash application as `Automobile Sales Statistics Dashboard`, center aligned, color `#503D36`, font size `24`.

## TASK 2.2 – Add drop-down menus to your dashboard with appropriate titles and options
- **Select Report-type Dropdown**: id `dropdown-statistics`, options `Yearly Statistics` and `Recession Period Statistics`, placeholder `Select a report type`, value `Select Statistics`
- **Select Year Dropdown**: id `select-year`, options are years 1980–2023, value/placeholder `Select-year`

## TASK 2.3 – Add a division for output display with appropriate id and classname property
Inner division: id `output-container`, className `chart-grid`, style `display: flex`.

In [2]:
# Set the title of the dashboard
app.title = "Automobile Sales Statistics Dashboard"

# Create the layout of the app
app.layout = html.Div([
    # TASK 2.1 Add title to the dashboard
    html.H1("Automobile Sales Statistics Dashboard", style={'textAlign': 'center', 'color': '#503D36', 'font-size': 24}),
    # TASK 2.2: Add two dropdown menus
    html.Div([
        html.Label("Select Statistics:"),
        dcc.Dropdown(
            id='dropdown-statistics',
            options=[
                {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
                {'label': 'Recession Period Statistics', 'value': 'Recession Period Statistics'}
            ],
            value='Select Statistics',
            placeholder='Select a report type',
            style={'width': '80%', 'padding': '3px', 'font-size': '20px', 'text-align-last': 'center'}
        )
    ]),
    html.Div(dcc.Dropdown(
        id='select-year',
        options=[{'label': i, 'value': i} for i in [y for y in range(1980, 2024)]],
        value='Select-year',
        placeholder='Select-year',
        style={'width': '80%', 'padding': '3px', 'font-size': '20px', 'text-align-last': 'center'}
    )),
    # TASK 2.3: Add a division for output display
    html.Div([
        html.Div(id='output-container', className='chart-grid', style={'display': 'flex'}),
    ])
])
print('Dashboard layout created.')

Dashboard layout created.


## TASK 2.4 – Creating Callbacks

### Update Input Container callback function
Controls the `disabled` property of the `select-year` dropdown based on the selected report type:
- Enabled (`False`) when `Yearly Statistics` is selected
- Disabled (`True`) when `Recession Period Statistics` is selected

In [3]:
# TASK 2.4: Define the callback function to update the input container based on the selected statistics
@app.callback(
    Output(component_id='select-year', component_property='disabled'),
    Input(component_id='dropdown-statistics', component_property='value'))

def update_input_container(selected_statistics):
    if selected_statistics == 'Yearly Statistics':
        return False
    else:
        return True

## TASK 2.5 – Create and display graphs for Recession Report Statistics

1. Average automobile sales fluctuation over the recession period (year-wise) – line chart
2. Average number of vehicles sold by vehicle type – bar chart
3. Total expenditure share by vehicle type during recessions – pie chart
4. Effect of unemployment rate on vehicle type and sales – bar chart

## TASK 2.6 – Create and display graphs for Yearly Report Statistics

1. Yearly automobile sales using line chart for the whole period
2. Total monthly automobile sales using line chart
3. Average number of vehicles sold by vehicle type in the selected year – bar chart
4. Total advertisement expenditure for each vehicle – pie chart

In [4]:
# Callback for plotting
@app.callback(
    Output(component_id='output-container', component_property='children'),
    [Input(component_id='dropdown-statistics', component_property='value'), Input(component_id='select-year', component_property='value')])

def update_output_container(selected_statistics, input_year):
    if selected_statistics == 'Recession Period Statistics':
        # Filter the data for recession periods
        recession_data = data[data['Recession'] == 1]

        # TASK 2.5: Create and display graphs for Recession Report Statistics

        # Plot 1: Automobile sales fluctuate over Recession Period (year-wise) using line chart
        yearly_rec = recession_data.groupby('Year')['Automobile_Sales'].mean().reset_index()
        R_chart1 = dcc.Graph(
            figure=px.line(yearly_rec,
                x='Year',
                y='Automobile_Sales',
                title="Average Automobile Sales fluctuation over Recession Period"))

        # Plot 2: Calculate the average number of vehicles sold by vehicle type and represent as a Bar chart
        average_sales = recession_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
        R_chart2 = dcc.Graph(
            figure=px.bar(average_sales,
                x='Vehicle_Type',
                y='Automobile_Sales',
                title="Average Number of Vehicles Sold by Vehicle Type during Recession"))

        # Plot 3: Pie chart for total expenditure share by vehicle type during recessions
        exp_rec = recession_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
        R_chart3 = dcc.Graph(
            figure=px.pie(exp_rec,
                values='Advertising_Expenditure',
                names='Vehicle_Type',
                title="Total Expenditure Share by Vehicle Type during Recessions"))

        # Plot 4: Bar chart for the effect of unemployment rate on vehicle type and sales
        unemp_data = recession_data.groupby(['unemployment_rate', 'Vehicle_Type'])['Automobile_Sales'].mean().reset_index()
        R_chart4 = dcc.Graph(figure=px.bar(unemp_data,
            x='unemployment_rate',
            y='Automobile_Sales',
            color='Vehicle_Type',
            labels={'unemployment_rate': 'Unemployment Rate', 'Automobile_Sales': 'Average Automobile Sales'},
            title='Effect of Unemployment Rate on Vehicle Type and Sales'))

        return [
            html.Div(className='chart-item', children=[html.Div(children=R_chart1), html.Div(children=R_chart2)], style={'display': 'flex'}),
            html.Div(className='chart-item', children=[html.Div(children=R_chart3), html.Div(children=R_chart4)], style={'display': 'flex'})
        ]

    # TASK 2.6: Create and display graphs for Yearly Report Statistics
    elif (input_year and selected_statistics == 'Yearly Statistics'):
        yearly_data = data[data['Year'] == input_year]

        # Plot 1: Yearly Automobile sales using line chart for the whole period
        yas = data.groupby('Year')['Automobile_Sales'].mean().reset_index()
        Y_chart1 = dcc.Graph(figure=px.line(yas,
            x='Year',
            y='Automobile_Sales',
            title='Yearly Automobile Sales'))

        # Plot 2: Total Monthly Automobile sales using line chart
        mas = yearly_data.groupby('Month')['Automobile_Sales'].sum().reset_index()
        Y_chart2 = dcc.Graph(figure=px.line(mas,
            x='Month',
            y='Automobile_Sales',
            title='Total Monthly Automobile Sales'))

        # Plot 3: Bar chart for average number of vehicles sold during the given year
        avr_vdata = yearly_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
        Y_chart3 = dcc.Graph(figure=px.bar(avr_vdata,
            x='Vehicle_Type',
            y='Automobile_Sales',
            title='Average Vehicles Sold by Vehicle Type in the year {}'.format(input_year)))

        # Plot 4: Total Advertisement Expenditure for each vehicle using pie chart
        exp_data = yearly_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
        Y_chart4 = dcc.Graph(
            figure=px.pie(exp_data,
                values='Advertising_Expenditure',
                names='Vehicle_Type',
                title='Total Advertisement Expenditure for Each Vehicle'))

        # TASK 2.6: Returning the graphs for displaying Yearly data
        return [
            html.Div(className='chart-item', children=[html.Div(children=Y_chart1), html.Div(children=Y_chart2)], style={'display': 'flex'}),
            html.Div(className='chart-item', children=[html.Div(children=Y_chart3), html.Div(children=Y_chart4)], style={'display': 'flex'})
        ]
    else:
        return None

## Demo – Render the dashboard figures in this notebook

Instead of launching the Dash server (which would block the kernel), we invoke the callback logic directly and display the resulting Plotly figures.

**Recession Period Statistics**

In [5]:
# Helper: extract plotly figures from the callback output container
import plotly.io as pio

def extract_graphs(node):
    out = []
    if isinstance(node, (list, tuple)):
        for x in node:
            out += extract_graphs(x)
    elif hasattr(node, 'children'):
        out += extract_graphs(node.children)
    elif node is not None and hasattr(node, 'figure'):
        out.append(node.figure)
    return out

# Recession report figures
recession_divs = update_output_container('Recession Period Statistics', None)
recession_figs = extract_graphs(recession_divs)
print('Recession charts:', len(recession_figs))
for f in recession_figs:
    pio.show(go.Figure(f))

**Yearly Statistics (year = 2010)**

In [6]:
# Yearly report figures
yearly_divs = update_output_container('Yearly Statistics', 2010)
yearly_figs = extract_graphs(yearly_divs)
print('Yearly charts:', len(yearly_figs))
for f in yearly_figs:
    pio.show(go.Figure(f))

Yearly charts: 4


## Run the Application

The application can be run as a standalone Python script (e.g. `python DV0101EN-Final-Assign-Part-2-Questions.py`). Inside this notebook the server is **not** started to avoid blocking the kernel.

```python
# Run the Dash app
if __name__ == '__main__':
    app.run(debug=True)
```

In [7]:
# Run the app (commented out in the notebook to avoid blocking the kernel)
# if __name__ == '__main__':
#     app.run(debug=True)

### Congratulations, you have successfully completed your application!

The dashboard displays two main reports:
- **Yearly Automobile Sales Statistics** (line charts for yearly/monthly sales, bar chart for vehicle-type sales, pie chart for advertisement expenditure)
- **Recession Period Statistics** (year-wise fluctuation line chart, vehicle-type average bar chart, expenditure share pie chart, unemployment effect bar chart)